# Phase 3 - Notebook 01: Qwen3-4B-Instruct LLM extraction on WDC-PAVE

**Settings:** Accelerator = GPU T4 x2, Internet = On. Run with *Save Version -> Save & Run All* (about 3-4 h).

Steps: integrity check -> GPU check -> download pinned Qwen3-4B-Instruct-2507 -> 10-product smoke test ->
4 prompting conditions (zero-shot, schema-guided, 1-shot, 3-shot) on the same 49 validation products ->
validation-only prompt selection -> all 354 test products -> comparison with CRF/BERT/DeBERTa using the same pair metric.
Output to download: `/kaggle/working/wdc_qwen_results.zip`.

In [ ]:
# ---- 1. Get the code: clone the project repository (Internet must be ON) ----
REPO_URL = "https://github.com/REPLACE_WITH_YOUR_USERNAME/nlp-product-attribute-extraction.git"
import os, subprocess, sys
from pathlib import Path
ROOT = Path("/kaggle/working/nlp-product-attribute-extraction")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts/phase3_llm")]
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---- 2. Install pinned packages and verify frozen data/code hashes ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-r", "requirements_kaggle_llm.txt"], check=True)
import kaggle_runner as runner
runner.configure(); runner.verify_bundle(); runner.check_data("wdc")

In [ ]:
# ---- 3. GPU check, then download the pinned model (~8 GB, kept outside saved output) ----
runner.preflight_gpu(); runner.download_model()

In [ ]:
# ---- 4. Smoke test (10 products) + four prompting conditions on 49 validation products ----
runner.run("wdc", "validation")

In [ ]:
from IPython.display import Markdown, display
_, val_dir, test_dir = runner.paths("wdc")
display(Markdown((val_dir / "summary.md").read_text()))

In [ ]:
# ---- 5. Freeze the best validation prompt and evaluate ALL 354 test products ----
runner.run("wdc", "test")
display(Markdown((test_dir / "summary.md").read_text()))

In [ ]:
# ---- 6. Results archive to download (attach to the repo under results/phase3_llm/) ----
print(runner.export_results("wdc"))